# Bayesian mixed-effects logistic regression — Pilot (Three)

Model: `correct ~ condition + (1|participant) + (1|question)`  
Family: Bernoulli (logit link)  
Reference condition: `birds` (control)  
Coefficients for `birds_repeat` and `birds_three` are log-odds differences vs control.

In [ ]:
import csv
from pathlib import Path

import arviz as az
import bambi as bmb
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

HERE = Path.cwd()
CSV_PATH = HERE / 'Prolific Reading-QA Pilot (Three)_June 5, 2026_15.13.csv'
YAML_PATH = HERE / 'birds_q.yaml'
TAG = 'bayes_three'

QIDS = [f'Q{n:02d}' for n in range(1, 11)]
LETTERS = ['A', 'B', 'C', 'D', 'E']
CONDS = ['birds', 'birds_repeat', 'birds_three']
COND_COLORS = {'birds': '#1f77b4', 'birds_repeat': '#ff7f0e', 'birds_three': '#2ca02c'}

In [ ]:
qdata = yaml.safe_load(YAML_PATH.read_text())['questions']
by_id = {q['q_id']: q for q in qdata}

with CSV_PATH.open(newline='', encoding='utf-8-sig') as f:
    rows = list(csv.DictReader(f))

ATTN_CHECKS = {'AT1': 'Gold versus green', 'AT2': 'Plumage strategy'}

def passed_attention(r):
    return all(r.get(k, '').strip() == v for k, v in ATTN_CHECKS.items())

finished_rows = [r for r in rows[2:] if r.get('Finished', '').lower() in {'true', '1'}]
data_rows = [r for r in finished_rows if passed_attention(r)]
by_cond = {c: [r for r in data_rows if r.get('assigned_doc') == c] for c in CONDS}
print('Retained respondents:', len(data_rows))
for c in CONDS:
    print(f'  {c}: {len(by_cond[c])}')

def edges(qid):
    return by_id[qid].get('metadata', {}).get('tags', {}).get('edges', 0) or 0

SORTED_QIDS = sorted(QIDS, key=edges)

In [ ]:
def resp_letters(r, qid):
    opts = by_id[qid]['options']
    text_to_letter = {v.strip(): k for k, v in opts.items()}
    val = r.get(qid, '').strip()
    if not val:
        return frozenset()
    parts = [p.strip() for p in val.split(',')]
    result = set()
    for p in parts:
        if p in LETTERS:
            result.add(p)
        elif p in text_to_letter:
            result.add(text_to_letter[p])
        else:
            m = next((k for k, v in opts.items() if v.strip().split() == p.split()), None)
            if m:
                result.add(m)
    return frozenset(result)

def correct_set(qid):
    ans = by_id[qid]['answer']
    return frozenset(ans if isinstance(ans, list) else [ans])

def is_correct(r, qid):
    return resp_letters(r, qid) == correct_set(qid)

In [ ]:
# Build long-format dataframe: one row per (participant, question) trial
records = []
for c in CONDS:
    for i, r in enumerate(by_cond[c]):
        pid = r.get('PROLIFIC_PID') or f'{c}_R{i}'
        for qid in SORTED_QIDS:
            records.append({
                'correct': int(is_correct(r, qid)),
                'condition': c,
                'participant': pid,
                'question': qid,
            })

df = pd.DataFrame(records)
# Set reference level: 'birds' (control)
df['condition'] = pd.Categorical(df['condition'], categories=CONDS, ordered=False)

print(df.shape)
print(df.groupby('condition')['correct'].agg(['mean', 'count']))

## Fit model

```
correct ~ condition + (1|participant) + (1|question)
```

Bambi uses weakly informative default priors (Normal(0,1) on log-odds scale for fixed effects, HalfNormal for random-effect SDs). With n≈8–10/cell this is appropriate — priors provide regularisation against overfit.

In [ ]:
model = bmb.Model(
    'correct ~ condition + (1|participant) + (1|question)',
    df,
    family='bernoulli',
)
model.build()
print(model)

In [ ]:
idata = model.fit(
    draws=2000,
    tune=1000,
    chains=4,
    target_accept=0.9,
    random_seed=42,
)
print(az.summary(idata, var_names=['condition'], round_to=3))

## Convergence diagnostics

In [ ]:
summary = az.summary(idata, var_names=['condition', 'Intercept'], round_to=3)
print(summary[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat', 'ess_bulk']])

## Posterior condition effects (log-odds and probability scale)

In [ ]:
# Extract posterior samples for condition coefficients
# post['condition'] has dims (chain, draw, condition_dim) with coords ['birds_repeat', 'birds_three']
post = idata.posterior

cond_levels = post['condition'].coords['condition_dim'].values.tolist()
print('Condition levels (vs birds reference):', cond_levels)

intercept = post['Intercept'].values.flatten()  # (chains*draws,)

def logistic(x):
    return 1 / (1 + np.exp(-x))

prob_samples = {'birds': logistic(intercept)}
coef_samples = {}
for label in cond_levels:
    coef = post['condition'].sel(condition_dim=label).values.flatten()
    coef_samples[label] = coef
    prob_samples[label] = logistic(intercept + coef)

condition_labels = ['birds'] + cond_levels

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: forest plot on log-odds scale
az.plot_forest(
    idata,
    var_names=['condition'],
    combined=True,
    hdi_prob=0.94,
    ax=axes[0],
)
axes[0].axvline(0, color='red', ls='--', alpha=0.7)
axes[0].set_title('Condition effects (log-odds vs birds control)\n94% HDI')
axes[0].set_xlabel('Log-odds difference')

# Right: P(correct) on probability scale
means = [prob_samples[c].mean() for c in condition_labels]
lo94  = [np.percentile(prob_samples[c], 3) for c in condition_labels]
hi94  = [np.percentile(prob_samples[c], 97) for c in condition_labels]

xs = np.arange(len(condition_labels))
colors = [COND_COLORS[c] for c in condition_labels]
axes[1].bar(xs, means, color=colors, alpha=0.8)
axes[1].errorbar(xs, means,
                 yerr=[np.array(means) - np.array(lo94), np.array(hi94) - np.array(means)],
                 fmt='none', color='black', capsize=5, linewidth=1.5)
axes[1].axhline(0.2, color='red', ls='--', label='chance (1/5)')
axes[1].set_xticks(xs)
axes[1].set_xticklabels(condition_labels, rotation=15, ha='right')
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('P(correct)')
axes[1].set_title('Posterior mean P(correct) per condition\n(marginalised over participant & question REs, 94% HDI)')
axes[1].legend()

plt.tight_layout()
fig.savefig(f'condition_effects_{TAG}.png', dpi=150, bbox_inches='tight')
plt.show()

## Posterior probability that each condition is worse than control

In [ ]:
for label in cond_levels:
    coef = coef_samples[label]
    p_worse  = (coef < 0).mean()
    p_better = (coef > 0).mean()
    lo, hi   = np.percentile(coef, [3, 97])
    prob_lo, prob_hi = np.percentile(prob_samples[label], [3, 97])
    print(f'{label} vs birds:')
    print(f'  log-odds mean={coef.mean():.2f}, 94% HDI [{lo:.2f}, {hi:.2f}]')
    print(f'  P(worse than control) = {p_worse:.2f},  P(better) = {p_better:.2f}')
    print(f'  P(correct) posterior mean={prob_samples[label].mean():.2f}, 94% HDI [{prob_lo:.2f}, {prob_hi:.2f}]')

## Random effect variance (participant vs question)

In [ ]:
re_vars = [v for v in post.data_vars if 'sigma' in v.lower() or '1|' in v]
print('Random effect parameters:', re_vars)
if re_vars:
    print(az.summary(idata, var_names=re_vars, round_to=3)[['mean', 'sd', 'hdi_3%', 'hdi_97%']])